# peaksMCP ARPES workspace

In [ ]:
import glob
import peaks as pks

folder = "/Users/haoxin/Desktop/112/实验数据/BP260623/data_netcdf"
files = sorted(glob.glob(f"{folder}/*.nc"))
print(f"找到 {len(files)} 个文件")
files

In [ ]:
import glob
import peaks as pks

folder = "/Users/haoxin/Desktop/112/实验数据/BP260623/data_netcdf"
files = sorted(glob.glob(f"{folder}/*.nc"))
print(f"找到 {len(files)} 个文件")
files

In [ ]:
data_all = pks.load(files)
data_all

In [ ]:
data_all = pks.load(files)
data_all

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

# 收集每个扫描的 2D 数据用于绘图；如果是 3D (含 deflector_perp)，先沿该维求和压缩成 2D
plot_list = []
plot_titles = []
for name, node in data_all.children.items():
    da = node.ds["data"]
    if "deflector_perp" in da.dims:
        da2d = da.sum(dim="deflector_perp")
    else:
        da2d = da
    plot_list.append(da2d)
    plot_titles.append(name)

print(f"共 {len(plot_list)} 个数据集将被绘制")

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

# 收集每个扫描的 2D 数据用于绘图；如果是 3D (含 deflector_perp)，先沿该维求和压缩成 2D
plot_list = []
plot_titles = []
for name, node in sorted(data_all.children.items()):
    da = node.ds["data"]
    if "deflector_perp" in da.dims:
        da2d = da.sum(dim="deflector_perp")
    else:
        da2d = da
    plot_list.append(da2d)
    plot_titles.append(name)

print(f"共 {len(plot_list)} 个数据集将被绘制，顺序：{plot_titles}")

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

# 收集每个扫描的 2D 数据用于绘图；如果是 3D (含 deflector_perp)，先沿该维求和压缩成 2D
plot_list = []
plot_titles = []
for name, node in sorted(data_all.children.items()):
    da = node.ds["data"]
    if "deflector_perp" in da.dims:
        da2d = da.sum(dim="deflector_perp")
    else:
        da2d = da
    plot_list.append(da2d)
    plot_titles.append(name)

print(f"共 {len(plot_list)} 个数据集将被绘制，顺序：{plot_titles}")

In [ ]:
import matplotlib.pyplot as plt
from peaks.core.display.plotting import plot_grid

plt.rcParams["figure.dpi"] = 80
plot_grid(plot_list, ncols=5, titles=plot_titles, figsize=(18, 18))
plt.savefig("/tmp/BP260623_all_scans.png", dpi=100, bbox_inches="tight")
print("已保存到 /tmp/BP260623_all_scans.png")

In [ ]:
import inspect
print(inspect.signature(plot_grid))
print(type(plot_grid))

In [ ]:
import matplotlib.pyplot as plt
from peaks.core.display.plotting import plot_grid

plt.rcParams["figure.dpi"] = 80
plot_grid(plot_list, ncols=5, titles=plot_titles, figsize=(18, 18))
plt.savefig("/tmp/BP260623_all_scans.png", dpi=100, bbox_inches="tight")
plt.close()
print("已保存到 /tmp/BP260623_all_scans.png")

In [ ]:
from peaks.core.display.plotting import plot_grid

plot_grid(plot_list, ncols=5, titles=plot_titles, figsize=(22, 22))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.close('all')

# 检查单个数据的实际数值范围，排查是否为全零/NaN
for i, (da, title) in enumerate(zip(plot_list[:5], plot_titles[:5])):
    vals = da.values
    print(title, "shape:", vals.shape, "min:", np.nanmin(vals), "max:", np.nanmax(vals),
          "nan_count:", np.isnan(vals).sum(), "all_zero:", np.all(vals == 0))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

plt.close('all')
plot_grid(plot_list, ncols=5, titles=plot_titles, figsize=(18, 18))
fig = plt.gcf()
fig.savefig("/tmp/BP260623_all_scans.png", dpi=100, bbox_inches="tight", facecolor="white")
plt.close('all')

img = Image.open("/tmp/BP260623_all_scans.png").convert("L")
arr = np.array(img)
print("图片尺寸:", img.size)
print("像素值范围: min=", arr.min(), "max=", arr.max(), "mean=", arr.mean())
print("是否全白(所有像素>250):", np.all(arr > 250))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# 临时切换到非交互式后端，避免在 notebook 里自动内联渲染出巨大的图（会超出输出限制）
prev_backend = matplotlib.get_backend()
matplotlib.use("Agg")

vmax_list = [float(np.nanpercentile(da.values, 99)) for da in plot_list]
vmin_list = [0.0 for _ in plot_list]

plot_grid(
    plot_list, ncols=5, titles=plot_titles, figsize=(20, 20),
    vmin=vmin_list, vmax=vmax_list, cmap="terrain"
)
fig = plt.gcf()
fig.savefig("/tmp/BP260623_all_scans.png", dpi=120, bbox_inches="tight", facecolor="white")
plt.close(fig)

# 切回 inline 后端，供后续单独查看小图使用
matplotlib.use(prev_backend)
print("已保存到 /tmp/BP260623_all_scans.png")

In [ ]:
from PIL import Image
import numpy as np

img = Image.open("/tmp/BP260623_all_scans.png").convert("L")
arr = np.array(img)
print("图片尺寸:", img.size)
print("像素均值:", arr.mean(), "（越低说明有效信号占比越大）")
print("是否近似全白(均值>250):", arr.mean() > 250)

In [ ]:
import numpy as np

# 检查每个数据集里非零像素的占比，验证"信号覆盖率"是否很低
for da, title in zip(plot_list, plot_titles):
    vals = da.values
    nonzero_frac = np.mean(vals > 0)
    print(f"{title}: 非零像素占比 = {nonzero_frac*100:.1f}%")